# 01 — Model Catalog Ingestion

Fetches model catalog metadata from any OpenAI-compatible /v1/models endpoint. Supports hybrid execution (headless cron jobs via environment variables or interactive UI widgets) with strict fail-fast validation and automated snapshot vaulting into datasets/.

## Ingestion Parameters

In [ ]:
provider_name = ''

In [ ]:
api_endpoint = ''

In [ ]:
api_key = ''

In [ ]:
save_model_data = True

## Fetch & Validate Catalog

In [ ]:
import os
import json
import requests
from datetime import datetime
from pathlib import Path

# 1. Hybrid Parameter Resolution
# Priority: UI Globals -> Explicit Env Vars -> Dynamic Provider Env Vars (e.g. FEATHERLESS_API_KEY)
raw_provider = globals().get("provider_name", "") or os.getenv("PROVIDER_NAME", "")
resolved_provider = str(raw_provider).strip()

raw_endpoint = globals().get("api_endpoint", "") or os.getenv("API_ENDPOINT", "")
resolved_endpoint = str(raw_endpoint).strip()

# Dynamic key lookup: direct input -> env var name specified -> API_KEY -> <PROVIDER>_API_KEY
raw_key = (
    globals().get("api_key", "") or
    (os.getenv(str(globals().get("api_key_env", "")).strip()) if globals().get("api_key_env") else "") or
    os.getenv("API_KEY", "") or
    (os.getenv(f"{resolved_provider.upper()}_API_KEY", "") if resolved_provider else "") or
    ""
)
resolved_key = str(raw_key).strip()

# Save toggle resolution
raw_save = globals().get("save_model_data", None)
if raw_save is None:
    env_save = os.getenv("SAVE_MODEL_DATA")
    resolved_save = (env_save.lower() in ("true", "1", "yes")) if env_save is not None else True
else:
    resolved_save = bool(raw_save)

# 2. Fail Fast: Validate Required Parameters with Diagnostic State
missing = []
if not resolved_provider:
    missing.append("provider_name")
if not resolved_endpoint:
    missing.append("api_endpoint")
if not resolved_key:
    missing.append("api_key (or API_KEY / <PROVIDER>_API_KEY env var)")

if missing:
    diag = (
        f"\n--- [DIAGNOSTIC STATE] ---\n"
        f"  globals().get('provider_name') = {repr(globals().get('provider_name'))}\n"
        f"  globals().get('api_endpoint')  = {repr(globals().get('api_endpoint'))}\n"
        f"  globals().get('api_key')       = {repr('***' if globals().get('api_key') else '')}\n"
        f"  os.environ.get('PROVIDER_NAME') = {repr(os.getenv('PROVIDER_NAME'))}\n"
        f"  os.environ.get('API_ENDPOINT')  = {repr(os.getenv('API_ENDPOINT'))}\n"
        f"---------------------------\n"
        f"Note: If you just typed into an input widget, ensure that the input widget block has executed "
        f"(or click 'Run All' / Shift+Enter on the input cells) so the Python kernel registers the new values."
    )
    raise ValueError(f"[FAIL FAST] Missing required configuration parameters: {', '.join(missing)}{diag}")

# 3. HTTP Request & Envelope Validation
url = f"{resolved_endpoint.rstrip('/')}/models"
headers = {"Authorization": f"Bearer {resolved_key}"}

print(f"[Ingestion] Fetching models for '{resolved_provider}' from: {url}")
res = requests.get(url, headers=headers, timeout=25)
res.raise_for_status()

try:
    payload = res.json()
except Exception as e:
    raise RuntimeError(f"[FAIL FAST] Response from {url} is not valid JSON: {res.text[:200]}") from e

# Validate /v1/models structure
if isinstance(payload, dict):
    models = payload.get("data")
    if models is None or not isinstance(models, list):
        raise RuntimeError(f"[FAIL FAST] Unexpected JSON structure from {url}. Expected 'data' key containing a list, got keys: {list(payload.keys())}")
elif isinstance(payload, list):
    models = payload
else:
    raise RuntimeError(f"[FAIL FAST] Unexpected payload type: {type(payload)}. Expected dict or list.")

print(f"[Ingestion] Successfully fetched {len(models)} models for provider '{resolved_provider}'.")

# 4. Storage & Snapshot Writer
if resolved_save:
    # Standard dataset naming: <provider>-models-<MMDDYY>.json (e.g. featherless-models-083126.json)
    date_str = datetime.now().strftime("%m%d%y")
    filename = f"{resolved_provider.lower()}-models-{date_str}.json"
    
    # Ensure datasets/ directory exists
    datasets_dir = Path("datasets")
    datasets_dir.mkdir(parents=True, exist_ok=True)
    output_path = datasets_dir / filename
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"[Snapshot Vault] Saved permanent dataset snapshot to: {output_path}")
else:
    # Ephemeral session output
    output_path = Path("models-list.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"[Ephemeral Session] Opted out of permanent snapshot. Saved session file to: {output_path}")
